In [2]:
reset()

_.<x> = QQ[]
F.<b> = (x^3-x^2-x-1).splitting_field(simplify_all=True)
R = F.ring_of_integers()
l = (x^3-x^2-x-1).roots(ring=R, multiplicities=False)
dl = l[2]/l[0]-l[1]/l[0]-l[2]/l[1]+l[0]/l[1]+l[1]/l[2]-l[0]/l[2]
c = [(1/l[1]-1/l[2])/dl, (1/l[2]-1/l[0])/dl, (1/l[0]-1/l[1])/dl]

# Sage's built-in power_mod doesn't work for us, so we implement our own version
def power_mod(base, exp, mod):
  I = F.ideal(mod)  # fractional, but integral, ideal generated by mod
  result = R(1)
  base = I.small_residue(R(base))
  while exp > 0:
    if exp % 2 == 1:  # If exp is odd
      result = I.small_residue(result * base)
    base = I.small_residue(base * base)
    exp //= 2
  return result

def trib(n, m): # Tribonacci number T_n mod m
  # we do a weird hack since the small_residue output can have a denominator of 22?
  t = c[0]*power_mod(l[0],n,m) + c[1]*power_mod(l[1],n,m) + c[2]*power_mod(l[2],n,m)
  return QQ(F.ideal(m).small_residue(t*22)) * inverse_mod(22,m) % m

def N_bound(p):
  _.<T> = GF(p)[]
  n_roots = len((T^3-T^2-T-1).roots(ring=GF(p), multiplicities=False))
  if n_roots == 3:
    return p-1
  elif n_roots == 1:
    return p^2-1
  elif n_roots == 0:
    return p^2+p+1
  else:
    raise ValueError(f"Unexpected number of roots for p = {p}: {n_roots}")

@parallel(max(os.cpu_count()-1,1))  # leave 1 core free for other tasks
def N0(p):
  if not p.is_prime():
    raise ValueError(f"Input to N0 must be prime, got {p}")
  n = N_bound(p)
  primes = n.prime_divisors()
  # we know that it works for n = N_bound(p), but we now repeatedly look at
  # the largest divisors of n to possibly replace it by a smaller number
  found = True
  while found:
    found = False
    for q in primes:
      # equivalent to if trib(n//q, p) == 0 and trib(n//q - 1, p) == 0:
      if (p*R).reduce(power_mod(R(l[0]/l[1]), n//q, p) - 1) == 0 and (p*R).reduce(power_mod(R(l[1]/l[2]), n//q, p) - 1) == 0:
        found = True
        if n % q^2 != 0:
          primes.remove(q)  # no need to check this prime again
        n //= q
        break
  print(f"N0({p}) = {n}", end=", ")
  with open("all.list", 'a') as f:
    print(f"N0({p}) = {n}", file=f)
  p21 = 1 if trib(n-1, p^2) == 0 else 0
  p22 = 1 if trib(n, p^2) == 0 else 0
  if p21 or p22:
    print(f"\np = {p}, n = {n}, {p21}{p22}")
    with open("stable.list", 'a') as f:
      print(f"p = {p}, n = {n}, {p21}{p22}", file=f)
  return (n,str(p21)+str(p22))

# trib(10, 17)

# serial
# [(p,N0(p)) for p in primes(start=13,stop=150)]
# parallel
[(p[0][0],q) for p,q in N0([p for p in primes(start=13, stop=10_000)])]

N0(13) = 56, N0(23) = 553, N0(17) = 96, N0(31) = 331, N0(19) = 120, N0(37) = 469, N0(47) = 46, N0(53) = 52, N0(29) = 140, N0(41) = 560, N0(43) = 308, N0(59) = 3541, N0(67) = 1519, N0(71) = 5113, N0(61) = 620, N0(73) = 1776, N0(89) = 8011, N0(97) = 3169, 
p = 97, n = 3169, 10
N0(79) = 1040, N0(83) = 287, N0(103) = 17, 
p = 103, n = 17, 01
N0(101) = 680, N0(113) = 12883, N0(107) = 1272, N0(131) = 5720, N0(127) = 1792, N0(137) = 18907, N0(109) = 330, N0(149) = 7400, N0(139) = 1288, N0(157) = 8269, N0(163) = 54, N0(167) = 9296, N0(179) = 32221, N0(181) = 10981, N0(151) = 950, N0(191) = 36673, N0(173) = 2494, N0(223) = 16651, N0(199) = 66, N0(197) = 3234, N0(193) = 1552, N0(211) = 1855, N0(227) = 17176, N0(229) = 17557, N0(251) = 63253, N0(257) = 256, N0(263) = 23056, N0(239) = 4760, N0(241) = 9680, N0(233) = 9048, N0(269) = 268, N0(271) = 24480, N0(311) = 310, N0(293) = 28616, N0(281) = 13160, N0(313) = 32761, N0(277) = 12788, N0(307) = 10472, N0(283) = 13348, N0(317) = 100807, N0(331) = 3

[(13, (56, '00')),
 (23, (553, '00')),
 (17, (96, '00')),
 (31, (331, '00')),
 (19, (120, '00')),
 (37, (469, '00')),
 (47, (46, '00')),
 (53, (52, '00')),
 (29, (140, '00')),
 (41, (560, '00')),
 (43, (308, '00')),
 (59, (3541, '00')),
 (67, (1519, '00')),
 (71, (5113, '00')),
 (61, (620, '00')),
 (73, (1776, '00')),
 (89, (8011, '00')),
 (97, (3169, '10')),
 (79, (1040, '00')),
 (83, (287, '00')),
 (103, (17, '01')),
 (101, (680, '00')),
 (113, (12883, '00')),
 (107, (1272, '00')),
 (131, (5720, '00')),
 (127, (1792, '00')),
 (137, (18907, '00')),
 (109, (330, '00')),
 (149, (7400, '00')),
 (139, (1288, '00')),
 (157, (8269, '00')),
 (163, (54, '00')),
 (167, (9296, '00')),
 (179, (32221, '00')),
 (181, (10981, '00')),
 (151, (950, '00')),
 (191, (36673, '00')),
 (173, (2494, '00')),
 (223, (16651, '00')),
 (199, (66, '00')),
 (197, (3234, '00')),
 (193, (1552, '00')),
 (211, (1855, '00')),
 (227, (17176, '00')),
 (229, (17557, '00')),
 (251, (63253, '00')),
 (257, (256, '00')),
 (26